# SCOPAtlas: Complete Workflow Tutorial

This notebook demonstrates the complete workflow for building a **Stable Operator Atlas** using scidiff.

## Overview

SCOPAtlas defines cellular states by their **local stability structure** rather than expression patterns alone. This provides a dynamical layer of cell identity that is invisible to expression-based atlases.

### Key Concepts

- **Operator Metrics**: Eigenvalue-derived measures of stability (λ_max⁺, λ_min⁻, P, S)
- **Operator Regimes**: Classification into stable, plastic, unstable, or deeply stable
- **Operator Embedding**: Low-dimensional projection of operator space
- **Operator Clustering**: Clustering based on dynamical properties

### Workflow

```
genes → latent (PCA/scVI) → drift model → J_ic → operator metrics → clustering
```

In [ ]:
# Import libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc

# Import scidiff modules
from scqdiff.models.drift import DriftField
from scqdiff.atlas import (
    StableOperatorAtlas,
    OperatorEmbedding,
    OperatorClustering,
    compute_operator_embedding,
    quick_operator_clustering
)

# Set plotting style
sc.set_figure_params(dpi=100, frameon=False)
sns.set_style('white')

# Random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Step 1: Load Data and Trained Model

Your data should have:
- **Latent representation**: `adata.obsm['X_pca']` or `adata.obsm['X_scvi']`
- **Pseudotime**: `adata.obs['pseudotime']` (optional)
- **Cell type annotations**: `adata.obs['cell_type']`
- **Condition labels**: `adata.obs['condition']` (optional)

In [ ]:
# Load your AnnData object
adata = ad.read_h5ad("path/to/your_data.h5ad")

# Load trained drift model
drift_model = torch.load("path/to/your_model.pt")

print(f"Loaded {adata.n_obs} cells with {adata.n_vars} genes")
print(f"Latent dimensions: {adata.obsm['X_pca'].shape[1]}")

## Step 2: Build Stable Operator Atlas

This computes operator metrics for each cell and classifies them into operator regimes.

In [ ]:
# Initialize atlas
atlas = StableOperatorAtlas(
    adata=adata,
    drift_model=drift_model,
    use_rep="X_pca",
    pseudotime_key="pseudotime",
    device="cpu"  # Use "cuda" if GPU available
)

# Build atlas
atlas.build(
    epsilon=0.1,
    threshold_unstable=0.1,
    threshold_plastic=0.05,
    threshold_deeply_stable=-1.0,
    plasticity_threshold=0.3,
    batch_size=32,
    compute_confidence=True
)

## Step 3: Examine Operator Metrics

The operator metrics are now stored in `adata.obs`.

In [ ]:
# View operator metrics
print("Operator metrics:")
print(adata.obs[['operator_regime', 'lambda_max_plus', 'lambda_min_minus', 
                 'plasticity', 'stable_dim']].head())

# Summary statistics
print("\nOperator regime distribution:")
print(adata.obs['operator_regime'].value_counts())

In [ ]:
# Plot distributions of operator metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = ['lambda_max_plus', 'lambda_min_minus', 'plasticity', 'stable_dim']
titles = ['Max Unstable Eigenvalue (λ_max⁺)', 'Stability Depth (λ_min⁻)', 
          'Plasticity Index (P)', 'Stable Subspace Dim (S)']

for ax, metric, title in zip(axes.flat, metrics, titles):
    sns.histplot(data=adata.obs, x=metric, hue='operator_regime', 
                kde=True, ax=ax, palette='Set2')
    ax.set_title(title)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

## Step 4: Compute Operator Embedding

Project operator metrics into low-dimensional space for visualization and clustering.

In [ ]:
# Compute operator embedding using metrics
operator_embedding = compute_operator_embedding(
    metrics=atlas.metrics,
    method='metrics',
    n_components=4
)

# Store in AnnData
adata.obsm['X_operator'] = operator_embedding

print(f"Operator embedding shape: {operator_embedding.shape}")

## Step 5: Visualize Operator Regimes

Visualize operator regimes on UMAP and compare with cell types.

In [ ]:
# Compute UMAP if not already done
if 'X_umap' not in adata.obsm:
    sc.pp.neighbors(adata, use_rep='X_pca')
    sc.tl.umap(adata)

In [ ]:
# Plot operator regimes vs cell types
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(adata, color='operator_regime', ax=axes[0], show=False, 
          title='Operator Regimes (Dynamical)', palette='Set2')
sc.pl.umap(adata, color='cell_type', ax=axes[1], show=False,
          title='Cell Types (Expression-based)')

plt.tight_layout()
plt.show()

In [ ]:
# Plot continuous operator metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sc.pl.umap(adata, color='lambda_max_plus', ax=axes[0,0], show=False,
          title='Max Unstable Eigenvalue', cmap='RdBu_r')
sc.pl.umap(adata, color='lambda_min_minus', ax=axes[0,1], show=False,
          title='Stability Depth', cmap='RdBu_r')
sc.pl.umap(adata, color='plasticity', ax=axes[1,0], show=False,
          title='Plasticity Index', cmap='viridis')
sc.pl.umap(adata, color='stable_dim', ax=axes[1,1], show=False,
          title='Stable Subspace Dimension', cmap='viridis')

plt.tight_layout()
plt.show()

## Step 6: Validate Non-Redundancy with Cell Types

**Critical validation**: Show that operator regimes provide information beyond cell types.

In [ ]:
# Test 1: Regime diversity within cell types
validation = atlas.validate_nonredundancy(
    celltype_key='cell_type',
    condition_key='condition'  # Optional
)

In [ ]:
# Visualize: Same cell type, different operator regimes
# Create cross-tabulation
crosstab = pd.crosstab(adata.obs['cell_type'], adata.obs['operator_regime'], 
                       normalize='index')

plt.figure(figsize=(10, 6))
sns.heatmap(crosstab, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Fraction'})
plt.title('Operator Regime Distribution Across Cell Types')
plt.xlabel('Operator Regime')
plt.ylabel('Cell Type')
plt.tight_layout()
plt.show()

print("\nKey observation: Same cell type can have different operator regimes!")

## Step 7: Operator-Based Clustering

Cluster cells in operator space and compare with expression-based clustering.

In [ ]:
# Initialize clustering
clusterer = OperatorClustering(adata)

# Prepare operator features
clusterer.prepare_operator_features()

# Cluster in operator space
clusterer.cluster_operator_space(resolution=1.0)

# Cluster in joint space (expression + operator)
clusterer.cluster_joint_space(alpha=0.5, resolution=1.0)

# Standard expression-based clustering for comparison
sc.pp.neighbors(adata, use_rep='X_pca')
sc.tl.leiden(adata, resolution=1.0, key_added='expression_clusters')

In [ ]:
# Visualize different clustering methods
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sc.pl.umap(adata, color='expression_clusters', ax=axes[0], show=False,
          title='Expression-Based Clustering')
sc.pl.umap(adata, color='operator_clusters', ax=axes[1], show=False,
          title='Operator-Based Clustering')
sc.pl.umap(adata, color='joint_clusters', ax=axes[2], show=False,
          title='Joint Clustering (α=0.5)')

plt.tight_layout()
plt.show()

## Step 8: Compare Clustering Quality

Quantitatively compare clustering methods using ARI and NMI.

In [ ]:
# Compare clustering quality
results = clusterer.compare_clustering_quality(
    methods={
        'Expression': 'expression_clusters',
        'Operator': 'operator_clusters',
        'Joint (α=0.5)': 'joint_clusters'
    },
    celltype_key='cell_type',
    compute_silhouette=True
)

In [ ]:
# Visualize clustering quality comparison
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_to_plot = ['ari', 'nmi', 'silhouette']
titles = ['Adjusted Rand Index', 'Normalized Mutual Information', 'Silhouette Score']

for ax, metric, title in zip(axes, metrics_to_plot, titles):
    if metric in results_df.columns:
        results_df[metric].plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
        ax.set_title(title)
        ax.set_ylabel('Score')
        ax.set_xlabel('')
        ax.set_xticklabels(results_df.index, rotation=45, ha='right')
        ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## Step 9: Grid Search for Optimal Alpha

Find the optimal balance between expression and operator features.

In [ ]:
# Grid search over alpha values
alpha_results, best_alpha = clusterer.grid_search_alpha(
    alphas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    celltype_key='cell_type'
)

In [ ]:
# Plot alpha vs performance
alpha_df = pd.DataFrame(alpha_results).T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

alpha_df['ari'].plot(ax=axes[0], marker='o', color='blue')
axes[0].axvline(best_alpha, color='red', linestyle='--', label=f'Best α={best_alpha}')
axes[0].set_xlabel('Alpha (Operator Weight)')
axes[0].set_ylabel('ARI')
axes[0].set_title('Clustering Performance vs Alpha')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

alpha_df['n_clusters'].plot(ax=axes[1], marker='s', color='green')
axes[1].set_xlabel('Alpha (Operator Weight)')
axes[1].set_ylabel('Number of Clusters')
axes[1].set_title('Number of Clusters vs Alpha')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Biological Interpretation

Interpret operator regimes in biological context.

In [ ]:
# Get regime statistics
regime_stats = atlas.get_regime_statistics()

# Create summary table
regime_summary = pd.DataFrame({
    'Regime': regime_stats.keys(),
    'Count': [s['count'] for s in regime_stats.values()],
    'λ_max⁺ (mean)': [s['lambda_max_mean'] for s in regime_stats.values()],
    'λ_min⁻ (mean)': [s['lambda_min_mean'] for s in regime_stats.values()],
    'Plasticity (mean)': [s['plasticity_mean'] for s in regime_stats.values()],
})

print("\nOperator Regime Summary:")
print(regime_summary.to_string(index=False))

In [ ]:
# Biological interpretation guide
print("""
═══════════════════════════════════════════════════════════════
BIOLOGICAL INTERPRETATION OF OPERATOR REGIMES
═══════════════════════════════════════════════════════════════

1. STABLE (λ_max⁺ ≤ 0, large S)
   → Terminal differentiation, homeostasis
   → Examples: Mature cell types, quiescent states
   → Resistant to perturbations

2. PLASTIC (λ_max⁺ ≈ 0, high P)
   → Progenitor states, decision points
   → Examples: Stem cells, multipotent progenitors
   → Many accessible directions

3. UNSTABLE (λ_max⁺ > threshold)
   → Transition states, bifurcations
   → Examples: Cells undergoing differentiation
   → Sensitive to perturbations

4. DEEPLY STABLE (very negative λ_min⁻)
   → Resistant states, locked-in fates
   → Examples: Senescent cells, exhausted immune cells
   → Difficult to reprogram

KEY INSIGHTS:
• Same cell type can have different operator regimes across conditions
• Operator regimes predict response to perturbations
• Reveals dynamical changes invisible to expression analysis
═══════════════════════════════════════════════════════════════
""")

## Step 11: Save Results

Save the atlas with all operator metrics and clustering results.

In [ ]:
# Save atlas results
atlas.save("scopatlas_results.h5ad")

print("Atlas saved to scopatlas_results.h5ad")
print("\nStored in AnnData:")
print("  adata.obs['operator_regime']: Regime labels")
print("  adata.obs['lambda_max_plus']: Max unstable eigenvalue")
print("  adata.obs['lambda_min_minus']: Stability depth")
print("  adata.obs['plasticity']: Plasticity index")
print("  adata.obs['stable_dim']: Stable subspace dimension")
print("  adata.obsm['X_operator']: Operator embedding")
print("  adata.obs['operator_clusters']: Operator-based clusters")
print("  adata.obs['joint_clusters']: Joint clusters")

## Summary

This notebook demonstrated the complete SCOPAtlas workflow:

1. ✅ Built Stable Operator Atlas from trained drift model
2. ✅ Computed per-cell operator metrics (λ_max⁺, λ_min⁻, P, S)
3. ✅ Classified cells into operator regimes
4. ✅ Computed operator embedding for visualization
5. ✅ Validated non-redundancy with cell types
6. ✅ Performed operator-based clustering
7. ✅ Compared clustering methods quantitatively
8. ✅ Optimized expression/operator balance
9. ✅ Interpreted results biologically

### Next Steps

- Test on multiple biological systems
- Validate with perturbation experiments
- Integrate with trajectory analysis (CellRank)
- Overlay with chromatin accessibility (ATAC-seq)
- Write paper!

### Key Findings to Report

1. **Non-redundancy**: Operator regimes are not redundant with cell types
2. **Improved clustering**: Joint clustering outperforms expression-only
3. **Biological relevance**: Operator regimes capture dynamical properties
4. **Novel insights**: Same cell type can have different stability structures